In [1]:
import pandas as pd
import warnings
from sklearn.preprocessing import OneHotEncoder,StandardScaler,MinMaxScaler,OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings(action='ignore')

df=pd.read_csv('data.csv')

df.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0


In [3]:
df['Vehicle_Age'].unique()

array(['> 2 Years', '1-2 Year', '< 1 Year'], dtype=object)

In [3]:
df['Previously_Insured']=df['Previously_Insured'].map({
    0:'No',
    1:'Yes'
})

In [2]:
one_hot_cat=['Vehicle_Age']
ordinal_cat=['Gender','Vehicle_Damage','Previously_Insured']
numeric_cat=['Age','Driving_License','Region_Code','Annual_Premium','Policy_Sales_Channel','Vintage']


In [4]:
df.drop(columns=['id'],inplace=True)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
X=df.drop(columns='Response')
y=df['Response']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [34]:
import numpy as np
neg=np.sum(y_train==0)
pos=np.sum(y_train==1)
spw=neg/pos

In [7]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
transformer=ColumnTransformer(
    transformers=[
        ('one_hot',OneHotEncoder(drop='first',handle_unknown='ignore'),one_hot_cat),
        ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),ordinal_cat)
    ],remainder='drop'
)



In [6]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import average_precision_score
model_dict={
    'RandomForest':RandomForestClassifier(n_estimators=250,max_depth=None,max_features='sqrt',class_weight='balanced',random_state=42),
    'xgboost': XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=7.14,
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10,
    reg_alpha=0.0,
    gamma=1,
    max_delta_step=1,
    random_state=42
)

}

for name,model in model_dict.items():

    model=Pipeline(steps=[
    ('transformer',transformer),
    ('model',model) ])
    model.fit(X_train,y_train)
    preds=model.predict_proba(X_test)[:,-1]
    y_preds=(preds>=0.30).astype(int)
    print(name.upper())
    report=classification_report(y_test,y_preds)
    print(confusion_matrix(y_test,y_preds))
    print(average_precision_score(y_test,y_preds))

    print(report)


NameError: name 'XGBClassifier' is not defined

In [8]:
model=Pipeline(steps=[
    ('transformer',transformer),
    ('model',RandomForestClassifier(n_estimators=250,max_depth=None,max_features='sqrt',class_weight='balanced',random_state=42)) ])

In [52]:
model.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'transformer', 'model', 'transformer__force_int_remainder_cols', 'transformer__n_jobs', 'transformer__remainder', 'transformer__sparse_threshold', 'transformer__transformer_weights', 'transformer__transformers', 'transformer__verbose', 'transformer__verbose_feature_names_out', 'transformer__one_hot', 'transformer__ordinal_encoder', 'transformer__one_hot__categories', 'transformer__one_hot__drop', 'transformer__one_hot__dtype', 'transformer__one_hot__feature_name_combiner', 'transformer__one_hot__handle_unknown', 'transformer__one_hot__max_categories', 'transformer__one_hot__min_frequency', 'transformer__one_hot__sparse_output', 'transformer__ordinal_encoder__categories', 'transformer__ordinal_encoder__dtype', 'transformer__ordinal_encoder__encoded_missing_value', 'transformer__ordinal_encoder__handle_unknown', 'transformer__ordinal_encoder__max_categories', 'transformer__ordinal_encoder__min_frequency', 'transformer__ordinal_e

In [54]:
import importlib.util
spec = importlib.util.find_spec("mlflow")
print(spec.origin)


c:\Users\Asus\Downloads\Vehicle-insurance-Project\vehicle\Lib\site-packages\mlflow\__init__.py


In [10]:
import optuna
from sklearn.model_selection import StratifiedKFold,cross_val_score
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment('Optuna Hypertuning')

def objective(trial):
    with mlflow.start_run(run_name=f'Trial_{trial.number}',nested=True):
        params = {
            "model__n_estimators": trial.suggest_int("n_estimators", 200, 1200),
            "model__max_depth": trial.suggest_int("max_depth", 3, 30),
            "model__min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "model__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "model__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "model__bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "model__class_weight": trial.suggest_categorical(
                "class_weight",
                ["balanced", "balanced_subsample", None]
            ),
            "model__random_state": 42,
            "model__n_jobs": -1
        }
        threshold = trial.suggest_float("threshold", 0.1, 0.4)
        model.set_params(**params)

        cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

        scores=cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring='average_precision'
        )
        mlflow.log_params(params)
        for i,s in enumerate(scores):
            mlflow.log_metric('fold_score',float(s),step=i)
        mlflow.log_metric('cv_mean',np.mean(scores))

        return np.mean(scores)
    
with mlflow.start_run(run_name='optuna_parent'):
    study=optuna.create_study(direction='maximize')
    study.optimize(objective,n_trials=15)

    # Log the best result on the parent run too
    mlflow.log_metric("best_score", float(study.best_value))
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.sklearn.log_model(model,name='model')




[I 2026-02-12 09:42:41,195] A new study created in memory with name: no-name-bd207bd0-8580-413f-8a28-9ed590ee8022
[W 2026-02-12 09:49:36,094] Trial 0 failed with parameters: {'n_estimators': 861, 'max_depth': 16, 'min_samples_split': 13, 'min_samples_leaf': 4, 'max_features': None, 'bootstrap': True, 'class_weight': 'balanced_subsample', 'threshold': 0.3036553849470245} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Asus\Downloads\Vehicle-insurance-Project\vehicle\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Asus\AppData\Local\Temp\ipykernel_4944\2518040396.py", line 29, in objective
    scores=cross_val_score(
        model,
    ...<3 lines>...
        scoring='average_precision'
    )
  File "c:\Users\Asus\Downloads\Vehicle-insurance-Project\vehicle\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwa

KeyboardInterrupt: 